# Goal Setting and Monitoring

The Goal Setting and Monitoring pattern enables agents to pursue a defined objective through an iterative **generate → evaluate → refine** loop. The agent doesn't stop at the first output — it measures success against explicit criteria and continues until the goals are met or a maximum iteration count is reached.

> **🧭 When to use this pattern — and how Flyte helps**
>
> Use goal setting when an agent must autonomously pursue a high-level objective across many steps, adapt as conditions change, and get there with minimal human steering. In Flyte, wrap the built-in loop in an `Agent` subclass, bound it with `max_iterations`, stream progress to a live `flyte.report` tab, and rely on task retries for transient failures.

## Implementation with Flyte v2 + the Agent harness

This refactor follows the docs' **"Strategy 1: wrap the built-in loop"**: a `GoalSeekingAgent` subclasses `Agent` and overrides `run`, delegating each generation to `super().run.aio(...)` and each judgement to a dedicated evaluator `Agent`. The outer generate → evaluate → refine loop lives in the subclass; the inner LLM ↔ tool turn loop is inherited.

#### LangChain vs Flyte v2 + Agent harness

| Aspect | LangChain | Flyte v2 + `Agent` harness |
|--------|-----------|----------------------------|
| **Outer loop** | Hand-written `while` over `goals_met()` | `GoalSeekingAgent.run` wrapping `super().run.aio` |
| **Generator / evaluator** | Two `ChatOpenAI` calls | Two `Agent`s (generator subclass + evaluator) |
| **LLM client** | LangChain wrapper | Harness' litellm callback |
| **Live progress** | `print()` | `flyte.report` HTML tab, updated each iteration |
| **Retries** | Manual `try/except` | `retries=3` on `@env.task` |
| **Secrets** | `.env` / `dotenv` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [15]:

import os
from dataclasses import dataclass, field
from datetime import timedelta

import flyte
import flyte.report
from flyte.ai.agents import Agent, AgentResult

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="goal-agent", python_version=(3, 12))
    .with_pip_packages("litellm")
)

goal_env = flyte.TaskEnvironment(
    name="goal_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the result model

`GoalAgentResult` is the typed, serializable output surfaced in the Flyte UI: the final code, how many iterations it took, and whether the agent converged (met all goals) before the cap.

In [16]:
@dataclass
class GoalAgentResult:
    """Final output of the goal-setting agent.

    Every field has an explicit primitive default. This matters because a
    notebook-defined dataclass is shipped to the cluster by *value* (cloudpickle),
    and after that round-trip a field with no default carries a fresh
    ``dataclasses.MISSING`` sentinel that fails the ``is MISSING`` identity check.
    mashumaro then writes that raw sentinel object into the JSON schema, and Flyte
    can't pack it into a protobuf Struct -> ``ValueError: Unexpected type`` at output
    conversion. Concrete defaults serialize cleanly and sidestep the issue.
    """
    final_code: str = ""
    iterations_used: int = 0
    converged: bool = False

### 5. Subclass `Agent` to wrap the loop

**Two agents, not one.** A *generator* writes the code; a separate *evaluator* judges it against the goals. Keeping them separate (as the LangChain example does) avoids self-serving rationalization: the evaluator sees only the code and the goals — never the generator's reasoning — so it can't talk itself into a pass.

**What the subclass does.** `GoalSeekingAgent` overrides `run` with the *outer* generate → evaluate → refine loop. Each iteration calls `super().run.aio(...)` to produce a candidate, then the evaluator `Agent` to judge it, and feeds the verdict back as feedback — until the evaluator says PASS or the loop hits `max_iterations`. The *inner* LLM ↔ tool loop is inherited unchanged from the base `Agent`.

**Why `run` is a plain `async def` (not `@syncify`).** In a notebook, tasks are submitted from `__main__`, so Flyte serializes them with cloudpickle *by value*. That would pickle this subclass — and a `@syncify`-wrapped method drags in its background-loop `SimpleQueue`, which isn't picklable (you'd get `cannot pickle '_queue.SimpleQueue'` at `flyte.run`). A plain `async def run`, called as `await goal_seeker.run(...)`, sidesteps it. The inherited `super().run.aio(...)` still works because the base `Agent` is a library class — serialized *by reference*, not by value.

In [17]:
GENERATOR_SYSTEM = """\
You are an expert Python developer.
Generate clean, idiomatic Python code that satisfies the stated use case and goals.
Return ONLY the raw Python source — no prose, no markdown fences."""

EVALUATOR_SYSTEM = """\
You are a strict code reviewer. Evaluate the provided Python code against the stated goals.
Be skeptical: do NOT pass code just because it looks plausible.
- Trace the function by hand on EVERY example in the use case. If any computed result
  differs from the stated expected value, respond FAIL.
- Verify the docstring's own examples match what the code actually returns.
- FAIL if the code contains markdown fences or any prose outside the source.
Respond with exactly TWO lines:
Line 1: PASS or FAIL
Line 2: One sentence of specific, actionable feedback (even for PASS). If FAIL, name a
concrete input and the wrong output it produces."""


def _esc(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _strip_fences(text: str) -> str:
    """Strip a leading ```/```python fence and trailing ``` so final_code is runnable.

    The generator is told not to emit fences, but models often do anyway — and the
    evaluator can fail on them, so unfenced code also reaches a clean PASS faster.
    """
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else ""
        if t.rstrip().endswith("```"):
            t = t.rstrip()[:-3]
    return t.strip()


@dataclass
class GoalSeekingAgent(Agent):
    """Wraps the built-in Agent loop with a generate -> evaluate -> refine outer loop."""
    max_iterations: int = 5
    evaluator: Agent | None = None

    async def run(self, message: str, history: list | None = None) -> AgentResult:
        feedback, previous = "", ""

        for i in range(self.max_iterations):
            gen_prompt = message
            if previous:
                gen_prompt += f"\n\nPrevious code (revise this):\n{previous}"
            if feedback:
                gen_prompt += f"\n\nFeedback to address:\n{feedback}"

            # Inner loop (LLM turns) is inherited from Agent.run.
            gen = await super(GoalSeekingAgent, self).run.aio(gen_prompt)
            previous = _strip_fences(gen.summary or "")

            verdict = await self.evaluator.run.aio(
                f"{message}\n\nCode to evaluate:\n{previous}"
            )
            lines = (verdict.summary or "").strip().splitlines()
            passed = bool(lines) and lines[0].strip().upper().startswith("PASS")
            feedback = lines[1].strip() if len(lines) > 1 else ""

            await flyte.report.log.aio(
                f"<h3>Iteration {i + 1} — {'PASS' if passed else 'FAIL'}</h3>"
                f"<p><em>{_esc(feedback)}</em></p><pre>{_esc(previous)}</pre>"
            )
            await flyte.report.flush.aio()

            if passed:
                return AgentResult(summary=previous, attempts=i + 1)

        return AgentResult(
            summary=previous,
            attempts=self.max_iterations,
            error="Reached max_iterations without meeting all goals.",
        )

### 6. Define the goal-setting task

 `GoalSeekingAgent.run` is a thin wrapper that builds the prompt and maps the `AgentResult` to a typed `GoalAgentResult`.

In [18]:
goal_seeker = GoalSeekingAgent(
    name="code-generator",
    model="claude-haiku-4-5",
    instructions=GENERATOR_SYSTEM,
    max_iterations=5,
    evaluator=Agent(
        name="code-reviewer",
        model="claude-haiku-4-5",
        instructions=EVALUATOR_SYSTEM,
    ),
)


@goal_env.task(
    retries=3,
    timeout=timedelta(minutes=15),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def goal_agent(
    use_case: str,
    goals: list[str],
    max_iterations: int = 5,
) -> GoalAgentResult:
    """Goal-setting agent: generate -> evaluate -> refine until PASS or max iterations.

    The outer loop is owned by GoalSeekingAgent.run; each iteration is logged live to
    the report tab in the Flyte UI.
    """
    goal_seeker.max_iterations = max_iterations
    prompt = f"Use case: {use_case}\n\nGoals:\n" + "\n".join(f"- {g}" for g in goals)

    result: AgentResult = await goal_seeker.run(prompt)
    return GoalAgentResult(
        final_code=result.summary,
        iterations_used=result.attempts,
        # AgentResult.error defaults to "" (not None); the loop only sets it when it
        # hits max_iterations. So "converged" == "no error was recorded".
        converged=not result.error,
    )

### 7. Run locally

In [19]:
USE_CASE = (
    "Write a Python function `binary_gap(n: int) -> int` that returns the length of the "
    "longest binary gap of a positive integer. A binary gap is the longest run of "
    "consecutive 0 bits surrounded by 1 bits on BOTH ends, in the binary representation "
    "of n (the standard Codility definition). Count the zeros — do NOT return the "
    "distance between the surrounding 1s (that would be one too many).\n\n"
    "The function must return exactly these values:\n"
    "  binary_gap(5)   == 1      # 101         -> one zero\n"
    "  binary_gap(9)   == 2      # 1001        -> two zeros\n"
    "  binary_gap(20)  == 1      # 10100       -> one zero (trailing zero excluded)\n"
    "  binary_gap(529) == 4      # 1000010001  -> gaps of 4 and 3, longest is 4\n"
    "  binary_gap(15)  == 0      # 1111        -> no gap\n"
    "  binary_gap(32)  == 0      # 100000      -> trailing zeros are not surrounded\n"
    "  binary_gap(1)   == 0"
)

GOALS = [
    "Returns exactly the outputs shown for every example input",
    "Counts the zeros in the gap (Codility definition) — must NOT return zeros + 1",
    "Trailing zeros that are not closed by a later 1 do not count (binary_gap(32) == 0)",
    "Raises ValueError for non-positive or non-integer inputs",
    "Docstring examples match the function's actual return values",
    "Returns raw Python source only — no markdown fences, no prose",
]

run = flyte.run(goal_agent, use_case=USE_CASE, goals=GOALS, max_iterations=4)
run.wait()
result: GoalAgentResult = run.outputs()[0]

print(f"Converged: {result.converged}")
print(f"Iterations used: {result.iterations_used}")
print("\n" + "=" * 60)
print(result.final_code)

> Building 1 image...

> Building image goal-agent for environment goal_agent

✓ Built image for environment goal_agent: localhost:30000/goal-agent:c0e0f4726ed0c807007057a3d5d61d37

Output()

Converged: False
Iterations used: 4

def binary_gap(n: int) -> int:
    """
    Returns the length of the longest binary gap of a positive integer.
    
    A binary gap is the longest run of consecutive 0 bits surrounded by 1 bits
    on BOTH ends, in the binary representation of n.
    
    Args:
        n: A positive integer
        
    Returns:
        The length of the longest binary gap (counting the zeros)
        
    Raises:
        ValueError: If n is not a positive integer
        
    Examples:
        >>> binary_gap(5)
        1
        >>> binary_gap(9)
        2
        >>> binary_gap(20)
        1
        >>> binary_gap(529)
        4
        >>> binary_gap(15)
        0
        >>> binary_gap(32)
        0
        >>> binary_gap(1)
        0
    """
    if not isinstance(n, int) or isinstance(n, bool):
        raise ValueError("Input must be a positive integer")
    if n <= 0:
        raise ValueError("Input must be a positive integer")
    
    binary_str = bin(n)[2: